# 04 — SHAP Interpretation & Business Insights
**Customer Churn Prediction — Banking**

Goal: Explain *why* the model predicts churn — translate model outputs into actionable business recommendations.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap
import pickle
import warnings
warnings.filterwarnings('ignore')
shap.initjs()

sns.set_style('whitegrid')

# Load artifacts
with open('../data/best_model.pkl', 'rb') as f:
    artifacts = pickle.load(f)
with open('../data/processed_data.pkl', 'rb') as f:
    data = pickle.load(f)

model = artifacts['model']
X_test = data['X_test']
y_test = data['y_test']
feature_names = data['feature_names']

print('Artifacts loaded. Computing SHAP values...')

## 1. Compute SHAP Values

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Convert test data to DataFrame for readability
X_test_df = pd.DataFrame(X_test, columns=feature_names)
print(f'SHAP values computed for {X_test_df.shape[0]} test customers.')

## 2. Global Feature Importance (SHAP Summary)

In [ ]:
# Mean absolute SHAP values per feature
shap_importance = pd.DataFrame({
    'Feature': feature_names,
    'Mean_SHAP': np.abs(shap_values).mean(axis=0)
}).sort_values('Mean_SHAP', ascending=True)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#E63946' if v > shap_importance['Mean_SHAP'].median() else '#457B9D'
          for v in shap_importance['Mean_SHAP']]
bars = ax.barh(shap_importance['Feature'], shap_importance['Mean_SHAP'],
               color=colors, edgecolor='white', linewidth=0.5)

ax.set_xlabel('Mean |SHAP Value| — Average Impact on Churn Prediction', fontsize=11)
ax.set_title('Feature Importance — What Drives Customer Churn?', fontsize=14, fontweight='bold')

high_patch = mpatches.Patch(color='#E63946', label='High importance')
low_patch = mpatches.Patch(color='#457B9D', label='Lower importance')
ax.legend(handles=[high_patch, low_patch], fontsize=10)

plt.tight_layout()
plt.savefig('../outputs/shap_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. SHAP Beeswarm Plot — Direction of Impact

In [ ]:
plt.figure(figsize=(11, 8))
shap.summary_plot(shap_values, X_test_df, show=False, plot_size=(11, 8))
plt.title('SHAP Summary — Direction & Magnitude of Each Feature on Churn',
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('../outputs/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print('Red = high feature value increases churn probability')
print('Blue = low feature value increases churn probability')

## 4. Top 3 Drivers — Deep Dive

In [ ]:
top3 = shap_importance.nlargest(3, 'Mean_SHAP')['Feature'].tolist()
print(f'Top 3 churn drivers: {top3}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, feature in enumerate(top3):
    feat_idx = feature_names.index(feature)
    axes[i].scatter(X_test_df[feature], shap_values[:, feat_idx],
                    c=shap_values[:, feat_idx], cmap='RdBu_r',
                    alpha=0.4, s=15)
    axes[i].axhline(0, color='black', linewidth=1, linestyle='--', alpha=0.5)
    axes[i].set_xlabel(feature, fontsize=11)
    axes[i].set_ylabel('SHAP Value (impact on churn)', fontsize=10)
    axes[i].set_title(f'Driver #{i+1}: {feature}', fontsize=12, fontweight='bold')

plt.suptitle('Top 3 Churn Drivers — Relationship with Churn Probability',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../outputs/top3_drivers.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Business Recommendations

In [ ]:
# High-risk segment: older, inactive, single-product customers in Germany
import pandas as pd

original_df = pd.read_csv('../data/Churn_Modelling.csv')
original_df.drop(columns=['RowNumber', 'CustomerId', 'Surname'], inplace=True)

# Define high risk segment
high_risk = original_df[
    (original_df['Age'] >= 45) &
    (original_df['IsActiveMember'] == 0) &
    (original_df['NumOfProducts'] == 1)
]

print('=== HIGH-RISK SEGMENT PROFILE ===')
print(f'Segment size: {len(high_risk):,} customers ({len(high_risk)/len(original_df)*100:.1f}% of base)')
print(f'Churn rate in segment: {high_risk["Exited"].mean()*100:.1f}%')
print(f'vs. Overall churn rate: {original_df["Exited"].mean()*100:.1f}%')
print()
print('Geography breakdown:')
print(high_risk.groupby('Geography')['Exited'].mean().sort_values(ascending=False).apply(lambda x: f'{x*100:.1f}%'))

## Business Recommendations

Based on SHAP analysis, the model identifies three primary levers for churn reduction:

---

### 1. 🎯 Age-Based Retention Campaign
**Finding:** Customers aged 45+ show significantly higher churn probability.

**Recommendation:** Design a dedicated retention program for the 45+ segment — premium service tiers, dedicated relationship managers, or loyalty rewards tied to tenure.

---

### 2. 📦 Cross-Sell Single-Product Customers
**Finding:** Customers holding only 1 product churn at nearly 3x the rate of 2-product customers.

**Recommendation:** Introduce targeted bundling offers (e.g., savings account + credit card package) within 90 days of onboarding. Each additional product significantly reduces churn probability.

---

### 3. 🔔 Reactivation Program for Dormant Accounts
**Finding:** Inactive members churn at ~2x the rate of active ones.

**Recommendation:** Trigger automated re-engagement workflow when a customer shows no login or transaction activity for 60+ days — personalized email, push notification, or proactive advisor outreach.

---

### 4. 🌍 Germany-Specific Strategy
**Finding:** German customers churn at ~32% vs France at ~16%.

**Recommendation:** Investigate root cause (pricing competitiveness, product fit, service quality) in the German market. Consider regional pricing or product adaptation.

---

**Model business value estimate:**
If the model correctly flags 70% of churners (based on recall), and the bank retains even 30% of those flagged through targeted intervention — assuming average customer LTV of €5,000 — that translates to ~**€2.1M retained value per 10,000 customers** per year.